
# 제품별 배치 프롬프트 생성 (옵션 C, v2: cluster/label/desc 주입)

- `persona_attributes_weighted.jsonl`의 `meta`에서 **cluster/label/desc**를 읽어
  각 페르소나 블록에 **클러스터 컨텍스트**를 삽입합니다.


In [ ]:

# =============================
# 0) CONFIG
# =============================
from pathlib import Path

PERSONA_JSONL = Path("/mnt/data/persona_attributes_weighted.jsonl")
PRODUCT_XLSX  = Path("/mnt/data/product_info (2).xlsx")
OUT_JSONL     = Path("/mnt/data/prompts_C.jsonl")
OUT_PREVIEW   = Path("/mnt/data/prompts_C_preview.json")

BATCH_SIZE = 10  # 5~30 권장
META_KEYS = ["cluster","label","desc"]

print("CONFIG loaded.")

In [ ]:

# =============================
# 1) Load data
# =============================
import json, pandas as pd

# Personas
personas = []
with open(PERSONA_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            personas.append(json.loads(line))

# Products
df_prod = pd.read_excel(PRODUCT_XLSX)

def get(d, k, default=""):
    return d[k] if k in d and pd.notna(d[k]) else default

products = []
for _, r in df_prod.iterrows():
    products.append({
        "product_id": get(r, "product_id", str(get(r, "id", ""))),
        "product_name": get(r, "product_name", get(r, "name", "")),
        "category": get(r, "category", ""),
        "features": get(r, "features", ""),
        "launch_ym": get(r, "launch_ym", ""),
        "price": get(r, "price", ""),
        "ad_model": get(r, "ad_model", ""),
    })

len(personas), len(products)

In [ ]:

# =============================
# 2) Helpers
# =============================
from typing import List, Dict, Any

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def format_attributes_for_prompt(attrs: Dict[str, Any]) -> str:
    lines = []
    for k, vw in attrs.items():
        v = vw.get("value", None)
        w = vw.get("weight", 0.0)
        v_str = "None" if v is None else str(v)
        lines.append(f"- {k}: {v_str} (w={w:.3f})")
    return "\n".join(lines[:40])

def persona_to_prompt_block(p: Dict[str, Any]) -> str:
    meta = p.get("meta", {}) or {}
    cluster = meta.get("cluster", "")
    label = meta.get("label", "")
    desc = meta.get("desc", "")
    cluster_block = f"- cluster: {cluster}\n- label: {label}\n- desc: {desc}" if (cluster or label or desc) else "- cluster: N/A"
    return f"""
[페르소나]
- id: {p.get('persona_key','')}
- 속성(가중치 합=1):
{format_attributes_for_prompt(p.get('attributes', {}))}
- 클러스터 컨텍스트:
{cluster_block}
""".strip()

def build_batch_prompt(product: Dict[str, Any], persona_batch: List[Dict[str, Any]]) -> str:
    return f"""
[역할]
당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다.
아래의 "제품 정보"와 "페르소나 목록"을 바탕으로, 각 페르소나마다
해당 제품의 구매자 페르소나를 **싱글 턴**으로 완결된 JSON 객체로 생성하세요.
각 페르소나는 서로 독립적이며, 서로의 정보에 영향을 주지 마세요.

[제품 정보]
- product_id: {product.get('product_id','')}
- 제품명: {product.get('product_name','')}
- 출시월(YYYY-MM): {product.get('launch_ym','')}
- 카테고리: {product.get('category','')}
- 주요 특징: {product.get('features','')}
- 기준 가격대(원): {product.get('price','')}
- 광고모델: {product.get('ad_model','')}

[페르소나 목록]
{ "\n\n".join(persona_to_prompt_block(p) for p in persona_batch) }

[규칙]
- '클러스터 컨텍스트'는 페르소나의 배경 지침으로만 사용합니다. 속성 가중치(합=1)와 충돌 시 '속성 가중치'를 우선합니다.
- 2024-07 ~ 2025-06 월별로 구매확률(prob 0~1)과 예상수량(qty 정수)을 제시합니다.
- 추석/설, 광고, 계절성을 반영합니다.
- **반드시 아래 JSON 스키마(JSON 배열)를 출력**하고, 불필요한 설명 문장은 출력하지 마세요.

[출력 스키마(JSON 배열)]
[
  {
    "persona_id": "p_{product.get('product_id','')}_{'{'}persona_key{'}'}",
    "product_id": "{product.get('product_id','')}",
    "segment_ref": "{'{'}persona_key{'}'}",
    "attributes": { "{'{'}속성명{'}'}": { "{'{'}value{'}'}": "<값>", "{'{'}weight{'}'}": <0~1> }, "...": "..." },
    "purchase_pattern": {
      "avg_purchase_prob": <0~1>,
      "avg_purchase_qty": <int>,
      "seasonality": {"추석": "+x%", "설": "+y%"},
      "promotion_effect": "광고모델 노출 시 +z%"
    },
    "forecast_12mo": {
      "2024-07": {"prob": <0~1>, "qty": <int>},
      "...": {},
      "2025-06": {"prob": <0~1>, "qty": <int>}
    }
  },
  ...
]
""".strip()

In [ ]:

# =============================
# 3) Build & save
# =============================
import json

records = []
for prod in products:
    for batch in chunked(personas, BATCH_SIZE):
        rec = {
            "product": prod,
            "personas": [{"persona_key": p.get("persona_key")} for p in batch],
            "prompt": build_batch_prompt(prod, batch)
        }
        records.append(rec)

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

from pathlib import Path
Path(OUT_PREVIEW).write_text(json.dumps(records[:1], ensure_ascii=False, indent=2), encoding="utf-8")
len(records), OUT_JSONL, OUT_PREVIEW